# Analiza dostępności i usług sklepów Żabka w Polsce

Autorzy:
- Jakub Rosiak 251620,
- Mateusz Kosowski 251558,
- Nikodem Nowak 251598

Nasz projekt koncentruje się na analizie rozmieszczenia sieci sklepów Żabka w Polsce, wykorzystując zbiór danych o lokalizacji prawie 10 tysięcy placówek (stan na rok 2024) oraz dane demograficzne z Narodowego Spisu Powszechnego 2021 (siatka kilometrowa GUS). Poprzez integrację danych punktowych z warstwą demograficzną, projekt pozwala na wyznaczenie obszarów o wysokim potencjale inwestycyjnym. Rezultatem prac jest zestaw interaktywnych wizualizacji oraz wniosków biznesowych wspierających procesy decyzyjne w zakresie ekspansji sieci.

### Cele projektu:
- <b>Prezentacja danych statystycznych sieci:</b>
    - Analiza struktury usług dodatkowych
    - Ranking województw i miast pod względem liczby placówek.
    - Badanie korelacji między liczbą mieszkańców a liczbą placówek.
- <b>Analiza przestrzenna i demograficzna:</b>
    - Wizualizacja rozmieszczenia sklepów na mapie Polski
    - Wizualizacja liczby mieszkańców przypadających na jeden sklep w siatce kilometrowej.
    - Oszacowanie liczby Polaków posiadających sklep w bezpośrednim sąsiedztwie (analiza dostępności w oparciu o siatkę).
- <b>Wnioski biznesowe:</b>
    - Wskazanie obszarów o najwyższym potencjale inwestystycyjnym




In [23]:
# Importy
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from scipy.spatial import cKDTree
from scipy import stats
from branca.element import MacroElement
from jinja2 import Template


# Konfiguracja wyglądu Seaborn i Matplotlib
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333',
    'grid.color': '#ddd',
    'text.color': '#222',
    'figure.figsize': (14, 8)
})

# Tworzenie katalogu na output
os.makedirs("output", exist_ok=True)

# Ignorujemy ostrzeżenia aby konsola była czysta
warnings.filterwarnings('ignore')

W naszym projekcie posługujemy się dwoma różnymi systemami współrzędnych przestrzennych (CRS) definiowanych przez standard EPSG.

<b>EPSG:4326</b> to globalny system układu współrzędnych geograficznych, w którym jednostką są stopnie (długość i szerokość geograficzna). W naszym projekcie służy on do:

   1. Wizualizacji interaktywnej: Jest to standardowy układ obsługiwany przez bibliotekę Folium
   2. Przechowywania surowych danych: Współrzędne sklepów w pliku wejściowym zapisane są jako szerokość (lat) i długość (lng).

Natomiast <b> EPSG:2180</b> to system prostokątnych współrzędnych płaskich zaprojektowany specjalnie dla obszaru Polski. W naszym projekcie służy on do:
   1. Precyzyjnych obliczeń odległości: Ponieważ jednostką w tym układzie jest metr, pozwala on na dokładne wyliczenie dystansu między mieszkańcami a najbliższym sklepem Żabka
   2. Złączeń przestrzennych (Spatial Join): Umożliwia poprawne przypisanie sklepów do komórek siatki populacyjnej GUS, która natywnie korzysta z tego formatu.
   3. Analizy zagęszczenia: Jest niezbędny do rzetelnego operowania na danych o populacji

In [24]:
# Wczytywanie danych
df_shops = pd.read_csv("data/zabka_shops.csv")
df_shops = df_shops[
    (df_shops['lat'].between(49, 55)) &
    (df_shops['lng'].between(14, 25))
].copy()
df_shops['services'] = df_shops['services'].fillna('').astype(str)

# Przygotowanie GeoDataFrame w układzie 4326 dla Folium
gdf_shops_4326 = gpd.GeoDataFrame(
    df_shops,
    geometry=gpd.points_from_xy(df_shops.lng, df_shops.lat),
    crs="EPSG:4326"
)

# # Siatka GUS z populacją
try:
    gdf_population = gpd.read_file("data/GRID_NSP2021_RES/GRID_NSP2021_RES.shp")
    gdf_population.set_crs(epsg=2180, allow_override=True, inplace=True)
    gdf_shops_2180 = gdf_shops_4326.to_crs(epsg=2180)
    print(f"✓ Siatka GUS: {len(gdf_population)} komórek")
except Exception as e:
    print(f"✗ Błąd wczytywania siatki GUS: {e}")
    raise SystemExit(1)


Liczba sklepów Żabka w Polsce (2024r.): 9755
Siatka wczytana. Układ: EPSG:2180


Teraz musimy dokonać pewnych obliczeń przestrzennych, potrzebnych do wykresów i warstw mapy.

In [ ]:
# Sklepy w komórkach siatki
joined = gpd.sjoin(gdf_shops_2180, gdf_population, how="inner", predicate="within")
shop_counts = joined['index_right'].value_counts()
gdf_population['shop_count'] = 0
gdf_population.loc[shop_counts.index, 'shop_count'] = shop_counts

# Dystans do najbliższej Żabki (KDTree - szybki)
shop_coords = np.column_stack([
    gdf_shops_2180.geometry.x,
    gdf_shops_2180.geometry.y
])
tree = cKDTree(shop_coords)
centroids = gdf_population.geometry.centroid
grid_coords = np.column_stack([centroids.x, centroids.y])
distances, _ = tree.query(grid_coords)
gdf_population['distance_m'] = distances

# Kategorie dystansu
bins = [0, 1000, 2000, 5000, np.inf]
labels = ['< 1km', '1-2km', '2-5km', '> 5km']
gdf_population['distance_cat'] = pd.cut(gdf_population['distance_m'], bins=bins, labels=labels)

# Ludzie na sklep (tylko tam gdzie są sklepy) - do choropleth
mask_shops = gdf_population['shop_count'] > 0
gdf_population['people_per_shop'] = np.nan
gdf_population.loc[mask_shops, 'people_per_shop'] = (
    gdf_population.loc[mask_shops, 'RES'] / gdf_population.loc[mask_shops, 'shop_count']
)

# Obszary niedostępne - duży potencjał inwestycyjny (>= 500 osób i > 1.5 km)
underserved = gdf_population[
    (gdf_population['RES'] >= 500) &
    (gdf_population['distance_m'] > 1500)
].copy()
underserved['priority'] = underserved['RES'] * underserved['distance_m'] / 1000
underserved = underserved.sort_values('priority', ascending=False)

Trochę statystyk podsumowujących analizę dostępności sklepów Żabka w Polsce.

In [ ]:
# Statystyki dostępności sklepów
total_pop = gdf_population['RES'].sum()
pop_within_1km = gdf_population[gdf_population['distance_m'] <= 1000]['RES'].sum()
pop_within_2km = gdf_population[gdf_population['distance_m'] <= 2000]['RES'].sum()
pop_within_5km = gdf_population[gdf_population['distance_m'] <= 5000]['RES'].sum()
far_pop = gdf_population[gdf_population['distance_m'] > 5000]['RES'].sum()
avg_distance = gdf_population['distance_m'].mean()
median_distance = gdf_population['distance_m'].median()

Dane i stałe potrzebne do wykresów statystycznych.

In [ ]:
# Przygotowanie danych do wykresów
services_exploded = df_shops['services'].str.split(',').explode().str.strip()
services_counts = services_exploded.value_counts()

legend_map = {
    'ZBC': 'Żabka Café', 'ODP': 'Odpiek Pieczywa', 'PAC': 'Paczki',
    'TER': 'Płatność Kartą', 'GSM': 'Doładowania', 'KPO': 'Karty Podarunkowe',
    'RAC': 'Rachunki', 'REJ': 'Rejestracja SIM', 'DEN': 'Usługi Energetyczne',
    'LOT': 'Lotto', 'BIH': 'Cashback', 'DKM': 'Karta Miejska'
}
services_names = [legend_map.get(code, code) for code in services_counts.index]

top_cities = df_shops['city'].value_counts().head(10)
voivodeships = df_shops['voivodeship'].value_counts()
pop_by_dist = gdf_population.groupby('distance_cat')['RES'].sum().reindex(labels, fill_value=0)

# Kolory dla dystansów
color_map = {'< 1km': '#2ecc71', '1-2km': '#f1c40f', '2-5km': '#e67e22', '> 5km': '#e74c3c'}
dist_colors = [color_map[label] for label in labels]

# Funkcja do dodawania etykiet na słupkach
def add_bar_labels(ax, fontsize=9, offset=0):
    for p in ax.patches:
        w = p.get_width()
        if w > 0:
            ax.text(w + offset, p.get_y() + p.get_height()/2, f'{int(w)}',
                    ha='left', va='center', fontsize=fontsize)

Tworzymy wykresy podsumowujące analizę

In [ ]:
fig = plt.figure(figsize=(18, 12), facecolor='white')
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Usługi dodatkowe
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x=services_counts.values, y=services_names, palette="viridis", ax=ax1)
ax1.set_title("Usługi dodatkowe w sklepach", fontweight='bold', fontsize=13)
ax1.set_xlabel("Liczba placówek")
add_bar_labels(ax1)

# 2. Województwa
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(x=voivodeships.values, y=voivodeships.index, palette="mako", ax=ax2)
ax2.set_title("Liczba sklepów wg województw", fontweight='bold', fontsize=13)
ax2.set_xlabel("Liczba sklepów")
add_bar_labels(ax2)

# 3. Top miasta
ax3 = fig.add_subplot(gs[1, 0])
sns.barplot(x=top_cities.values, y=top_cities.index, palette="rocket", ax=ax3)
ax3.set_title("Top 10 miast", fontweight='bold', fontsize=13)
ax3.set_xlabel("Liczba sklepów")
add_bar_labels(ax3)

# 4. Populacja wg dystansu
ax4 = fig.add_subplot(gs[1, 1])
pop_by_dist.plot(kind='bar', ax=ax4, color=dist_colors)
ax4.set_title("Populacja wg dystansu do Żabki", fontweight='bold', fontsize=13)
ax4.set_xlabel("Dystans")
ax4.set_ylabel("Liczba mieszkańców")
ax4.set_xticklabels(labels, rotation=0)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1e6)}M'))

# 5. Krzywa pokrycia
ax5 = fig.add_subplot(gs[2, 0])
pop_data = gdf_population[gdf_population['RES'] > 0].copy()
pop_data = pop_data.sort_values('distance_m') / 1000
pop_data['cum_pop'] = pop_data['RES'].cumsum()
pop_data['cum_share'] = pop_data['cum_pop'] / pop_data['RES'].sum()
ax5.plot(pop_data['distance_m'] / 1000, pop_data['cum_share'], color='#4c78a8', linewidth=2.5)
for d in [1, 2, 5]:
    ax5.axvline(d, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax5.set_xlim(0, 10)
ax5.set_ylim(0, 1)
ax5.set_title("Krzywa pokrycia populacji", fontweight='bold', fontsize=13)
ax5.set_xlabel("Dystans do Żabki (km)")
ax5.set_ylabel("Udział populacji")
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x*100)}%'))

# 6. Scatter - ludność vs dystans
ax6 = fig.add_subplot(gs[2, 1])
sample = gdf_population[gdf_population['RES'] > 0].sample(
    min(5000, len(gdf_population)), random_state=42
)
ax6.scatter(sample['distance_m'] / 1000, sample['RES'], alpha=0.3, s=8, color='#72b7b2')
ax6.axvline(1.5, color='#e45756', linestyle='--', linewidth=1.5, label='Próg 1.5 km')
ax6.axhline(500, color='#f58518', linestyle='--', linewidth=1.5, label='Próg 500 osób')
ax6.set_xlim(0, 10)
ax6.set_title("Ludność vs dystans (próbka 5000)", fontweight='bold', fontsize=13)
ax6.set_xlabel("Dystans (km)")
ax6.set_ylabel("Ludność w komórce")
ax6.legend(frameon=True, loc='upper right')

plt.savefig('output/analiza_kompletna.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()




fig2 = plt.figure(figsize=(18, 6), facecolor='white')
gs2 = fig2.add_gridspec(1, 2, wspace=0.2)

# 7. HISTOGRAM DYSTANSÓW
ax7 = fig2.add_subplot(gs2[0, 0])
pop_with_res = gdf_population[gdf_population['RES'] > 0]
ax7.hist(pop_with_res['distance_m'] / 1000, bins=50, weights=pop_with_res['RES'],
         color='#4c78a8', edgecolor='white', alpha=0.8)
ax7.axvline(1, color='#2ecc71', linestyle='--', linewidth=2, label='1 km')
ax7.axvline(2, color='#f1c40f', linestyle='--', linewidth=2, label='2 km')
ax7.axvline(5, color='#e74c3c', linestyle='--', linewidth=2, label='5 km')
ax7.set_title("Rozkład dystansów do najbliższej Żabki", fontweight='bold', fontsize=13)
ax7.set_xlabel("Dystans (km)")
ax7.set_ylabel("Liczba mieszkańców")
ax7.set_xlim(0, 15)
ax7.legend(loc='upper right', frameon=True)
ax7.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1e6)}M'))

# 8. KORELACJA: GĘSTOŚĆ VS SKLEPY
ax8 = fig2.add_subplot(gs2[0, 1])
# Agregacja per komórka - ile mieszkańców na km2 vs ile sklepów w promieniu 1km
grid_with_pop = gdf_population[gdf_population['RES'] > 100].copy()
sample_corr = grid_with_pop.sample(min(3000, len(grid_with_pop)), random_state=42)

ax8.scatter(sample_corr['RES'], sample_corr['shop_count'], alpha=0.4, s=15, color='#72b7b2')

# Linia trendu
mask_valid = (sample_corr['shop_count'] > 0) & (sample_corr['RES'] > 0)
if mask_valid.sum() > 10:
    slope, intercept, r_value, _, _ = stats.linregress(
        sample_corr.loc[mask_valid, 'RES'],
        sample_corr.loc[mask_valid, 'shop_count']
    )
    x_line = np.linspace(sample_corr['RES'].min(), sample_corr['RES'].max(), 100)
    y_line = slope * x_line + intercept
    ax8.plot(x_line, y_line, color='#e45756', linewidth=2, label=f'R² = {r_value**2:.2f}')
    ax8.legend(loc='upper right', frameon=True)

ax8.set_title("Korelacja: liczba mieszkańców vs sklepy", fontweight='bold', fontsize=13)
ax8.set_xlabel("Liczba mieszkańców w komórce")
ax8.set_ylabel("Liczba sklepów")

plt.tight_layout()
plt.savefig('output/analiza_rozszerzona.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()